In [1]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Works when the notebook starts in notebooks/ or the project root
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

model_dir = (
    project_root / "kaggle" / "kaggle-output-retry" / "bert_sst2_best"
)

tokenizer = AutoTokenizer.from_pretrained(
    model_dir, local_files_only=True
)
model = AutoModelForSequenceClassification.from_pretrained(
    model_dir, local_files_only=True
)
model.eval()

print("Fine-tuned BERT loaded successfully!")

/Users/farabi/Farabi/ML-Projects/LLM/SST2-Sentiment/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12027.49it/s]

Fine-tuned BERT loaded successfully!


In [2]:
import torch

sentences = [
    "This movie was wonderful. I loved every minute!",
    "The movie was boring and a complete waste of time.",
]

inputs = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

with torch.inference_mode():
    probabilities = model(**inputs).logits.softmax(dim=-1)

# SST-2: 0 = negative, 1 = positive
label_names = ["negative", "positive"]

for sentence, scores in zip(sentences, probabilities):
    predicted_id = scores.argmax().item()
    print(f"Sentence: {sentence}")
    print(f"Prediction: {label_names[predicted_id]}")
    print(f"Model confidence: {scores[predicted_id].item():.2%}\n")

Sentence: This movie was wonderful. I loved every minute!
Prediction: positive
Model confidence: 99.95%

Sentence: The movie was boring and a complete waste of time.
Prediction: negative
Model confidence: 99.12%



In [3]:
from datasets import load_dataset

official_val = load_dataset(
    "stanfordnlp/sst2",
    split="validation",
)

print("Validation examples:", len(official_val))
print(official_val[0])

Validation examples: 872
{'idx': 0, 'sentence': "it 's a charming and often affecting journey . ", 'label': 1}


In [4]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)
model = model.to(device)
model.eval()

predictions = []
labels = []
batch_size = 16

with torch.inference_mode():
    for start in tqdm(range(0, len(official_val), batch_size)):
        batch = official_val[start:start + batch_size]

        inputs = tokenizer(
            batch["sentence"],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)

        predicted = model(**inputs).logits.argmax(dim=-1)

        predictions.extend(predicted.cpu().tolist())
        labels.extend(batch["label"])

print("Device:", device)
print(f"Official validation accuracy: {accuracy_score(labels, predictions):.4f}")
print(
    f"Official validation Macro F1: "
    f"{f1_score(labels, predictions, average='macro'):.4f}"
)

100%|██████████| 55/55 [00:03<00:00, 14.67it/s]

Device: mps
Official validation accuracy: 0.9117
Official validation Macro F1: 0.9116
